# 110장 · 응답 지연 시간 계산과 호출 비용의 측정 조건

지연 시간은 어느 시점부터 어디까지 재는지에 따라 달라집니다. 처리 구간과 사용자 대기 구간을 구별하고 호출 수와 비용의 측정 근거를 확인합니다.

한 요청의 구간별 지연과 외부 호출 비용을 기록합니다. 합성 시각의 차이, 실제 측정값, 미측정 항목을 분리해 서로 다른 실행을 비교할 수 있게 합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 팀별 변형을 시도한 뒤 해당 셀 아래에서 예시를 펼쳐 비교할 수 있습니다.


## 권한을 포함한 캐시 키로 원문 조회 비교하기

109장의 공개 오류에는 내부 자료가 없습니다. 정상 응답의 원문 조회만 캐시 대상으로 골라 같은 문서·판본·공개 범위에서 두 호출의 답을 비교합니다.

이 프로젝트의 노트북은 한 폴더에서 열어 주세요. 첫 셀은 `team-project/final/chapter109.json`을 먼저 찾습니다. 팀 파일이 없으면 [이 장의 시작 예시](chapter110-start.json)를 노트북 옆에 내려받으세요. 출력 첫 줄에서 어느 파일을 읽었는지 확인할 수 있습니다.

In [ ]:
from pathlib import Path
import json
import re

here = Path.cwd()
project_root = here.parent if re.fullmatch(r"chapter[0-9]+", here.name) else here
team_file = project_root / "team-project/final/chapter109.json"
sample_file = here / "chapter110-start.json"
input_file = team_file if team_file.is_file() else sample_file
if not input_file.is_file():
    raise FileNotFoundError("이 장의 시작 예시를 내려받아 노트북 옆에 두세요.")
incoming = json.loads(input_file.read_text(encoding="utf-8"))
print("팀 기록" if input_file == team_file else "교재 예시", input_file.name)
print(incoming['visible_keys'])

교재 예시 chapter110-start.json
['code', 'message']


두 번째 줄은 읽은 파일의 `visible_keys` 값입니다.

## 처리 시간과 대기 시간의 측정 구간

측정 카드 예시　Q01 / rules-v1 / 공개 그룹 / 시작은 서비스 처리 직전 / 종료는 결과 검증과 JSON 저장이 끝난 시점 / 업무 결과 answered / 근거 RULE-A. 실제 사용자 대기는 브라우저 전송과 표시 시간까지 더 관찰해야 합니다. 아래 숫자는 단위를 연습하기 위해 만든 합성 시각입니다.

**대기 시간과 전체 응답 시간**　처리 시간이 3.5 ms로 같아도 대기 4 ms를 포함한 전체 시간은 7.5 ms입니다.

[웹 교재에서 열기](https://langchain-lecture.vercel.app/chapter110/index.html#practice-space)


### 셀 01 · 합성 타임스탬프의 구간 차이

두 값은 같은 시계의 나노초 눈금입니다. 날짜처럼 해석하지 말고 차이를 구해 보세요.

실패 시도의 기록 예는 ‘종료 사건 timeout / 성공 여부 False / 브라우저 대기 시간 미측정’입니다. Q01의 정상 완료 기록과 구분하세요. 관찰하지 못한 시간 칸에 0을 적으면 즉시 끝났다는 뜻으로 읽힙니다.


In [1]:
started_ns = 12_000_000
finished_ns = 15_500_000
elapsed_ms = (finished_ns - started_ns) / 1_000_000
print(elapsed_ms, "ms (합성 시각)")

3.5 ms (합성 시각)


3.5ms는 이 두 고정 숫자의 차이입니다. 팀이 실제 시간을 잴 때는 같은 장비와 자료, 같은 시작·종료 위치에서 여러 번 관찰하고 실패한 시도도 남깁니다.

3.5ms는 예제에 미리 넣은 두 시각의 차이입니다. 실제 서비스의 응답 지연을 보고하려면 같은 요청에서 시작과 종료 시각을 관측해야 합니다. 호출 비용도 외부 호출 횟수, 실제 사용량, 적용 단가가 있어야 계산할 수 있습니다.


#### 3.5ms가 뜻하는 측정 구간

<mark>합성 시각의 차로 구한 값은 이 예시의 구간 길이입니다.</mark> 시작점이 요청 접수인지 실제 처리 시작인지에 따라 대기 시간이 포함되는지가 달라집니다. 팀 측정값에는 두 시점과 단위를 남겨 서로 다른 구간을 비교하지 않게 하세요.

선수 개념 · 49장 검색·인용 품질과 비용 측정 조건

#### 응답 시간의 측정 구간

$$t_{전체}=(t_{시작}-t_{접수})+(t_{종료}-t_{시작})$$

요청 접수부터 처리 시작까지는 대기 시간입니다. 처리 시작부터 종료까지 잰 3.5ms에 대기 시간을 더하면 같은 요청의 전체 시간이 됩니다.

### 두 눈금 사이의 차이를 밀리초로 읽기

합성 시각 15,500,000에서 12,000,000을 빼면 차이는 3,500,000나노초입니다. 1밀리초가 1,000,000나노초이므로 결과는 3.5밀리초입니다. 브라우저가 표시를 끝낸 시각까지 재지 않았다면 그 시간은 이 차이에 들어가지 않습니다.


## 성능 변경 전후 답변 품질 대조

비교표 한 행의 예시　대상 Q01 / 가설 “같은 공개 문서의 읽기 준비를 줄일 수 있다” / 고정 기준 본문 전원 표시등, RULE-A·2.1절·판본·허용 범위 / 바꾸려는 부분 문서 준비 횟수 / 결정 실측 전 보류. 현재 예제는 외부 모델 호출 0회입니다. 측정하지 않은 운영비와 청구 토큰을 비용 0원으로 적을 근거는 없습니다.


### 팀 참고 예시 · 호출 비용의 측정 조건


In [2]:
record = {"model_calls": 0, "actual_operating_cost": None, "source": "로컬 규칙 실행"}
print(record["source"], record["model_calls"], record["actual_operating_cost"])

로컬 규칙 실행 0 None


0은 관찰한 모델 호출 수입니다. None은 운영비를 측정하지 않았다는 뜻입니다. 팀에 청구서가 생기면 기간·통화·포함 항목부터 확인하세요.


## 같은 답을 유지하면서 문서를 준비한 횟수 비교하기

Q01을 연달아 물을 때마다 같은 규정을 다시 읽습니다. 준비한 문서를 잠시 재사용하면 읽기 횟수가 줄어드는지 측정하고, 답과 근거가 동일한지도 함께 확인합니다.

### 셀 04 · 원문 읽기 함수에 카운터 달기

아래 자료는 한 판본의 공개 규정 한 건입니다. 반복 측정을 위한 로컬 실험으로 문서 ID·판본·그룹을 키에 넣습니다. 실제 지연 시간이나 모델 요금은 이 횟수에서 추정하지 않습니다.

In [3]:
from functools import lru_cache

rule_rows = {("RULE-A", "rules-v1", "public"): ("점검 전 전원 표시등을 확인합니다.", "2.1")}
read_log = []
def read_rule(doc_id, version, group):
    key = (doc_id, version, group)
    read_log.append(key)
    return rule_rows[key]

key = ("RULE-A", "rules-v1", "public")
baseline = [read_rule(*key) for _ in range(2)]
uncached_reads = len(read_log)
read_log.clear()
cached_read = lru_cache(maxsize=2)(read_rule)
candidate_answers = [cached_read(*key) for _ in range(2)]
print("원문 읽기", uncached_reads, len(read_log))
print("문장과 절 일치", candidate_answers == baseline)

원문 읽기 2 1
문장과 절 일치 True


입력은 두 번 들어왔지만 후보의 원문 읽기는 한 번입니다. 결과의 문장과 절이 같으므로 이 입력에서는 읽기 작업을 줄였습니다. 다른 판본이나 권한 규칙이 바뀌면 캐시를 무효화하는 조건도 필요합니다.

### 팀 측정표에 넣을 단위 고르기

실제 지연을 비교하려면 첫 호출과 캐시가 찬 다음 호출을 나누고 같은 질문을 반복 측정하세요. 여기서 측정한 것은 읽기 횟수입니다. “원문 읽기 2회에서 1회”는 관찰한 값이고 “사용자 응답이 두 배 빨라졌다”는 별도 측정이 필요합니다.

<mark>준비 횟수가 줄었다면 같은 자료와 권한 범위에서 결과도 보존됐는지 확인합니다.</mark> 그룹을 키에서 빼면 접근 범위가 다른 사용자가 같은 결과를 재사용할 수 있습니다. 판본을 구분해도 권한 정책 변경 시 별도 무효화가 필요합니다.

111장에서는 수정한 기준, 기존에 맞던 입력, 보존해야 할 반례를 한 회귀 묶음으로 실행합니다.

## 팀 사례에 적용하기 · 권한을 포함한 캐시 키로 원문 조회 비교하기

위 예제에 첫 셀에서 읽어 둔 팀 기록을 넣어 봅니다.

In [ ]:
from functools import lru_cache
source_map = {(d["id"], d["version"], "public"): d["text"] for d in incoming["reply"]["sources"]}
@lru_cache(maxsize=4)
def cached_text(doc_id, version, group):
    return source_map.get((doc_id, version, group))
answers = []
for key in source_map:
    answers.append(cached_text(*key) == cached_text(*key) == source_map[key])
new = {**incoming, "cache_check": {"same": bool(answers) and all(answers),
       "hits": cached_text.cache_info().hits, "misses": cached_text.cache_info().misses}}
print(new["cache_check"])

{'same': True, 'hits': 1, 'misses': 1}


같은 키를 두 번 읽어도 원문 준비는 한 번입니다. 그룹이나 판본이 달라진 요청은 별도 키로 취급하고 매칭되는 원문이 없으면 반환하지 않습니다.

In [ ]:
output_file = project_root / "team-project/final/chapter110.json"
output_file.parent.mkdir(parents=True, exist_ok=True)
with output_file.open("a+", encoding="utf-8") as handle:
    handle.seek(0)
    previous_text = handle.read()
    if previous_text and json.loads(previous_text) != new:
        raise FileExistsError("같은 장의 다른 결과가 있습니다. 기존 JSON을 다른 이름으로 보관한 뒤 다시 실행하세요.")
    if not previous_text:
        json.dump(new, handle, ensure_ascii=False, indent=2)
        handle.write("\n")
print(output_file.relative_to(project_root).as_posix())

team-project/final/chapter110.json


111장 첫 셀이 이 파일을 읽습니다. 입력을 바꿔 다시 실행할 때는 기존 결과 파일의 이름을 바꿔 보관하세요.

## 돌아보기

111장에는 측정 구간, 자료·질문 버전, 같은 결과를 보장하는 기준을 함께 넘기세요. 실제 HTTP 지연이나 청구서를 확보하기 전에는 운영 시간과 비용을 빈칸으로 둡니다.
